## Algorithmic Game Theory, Fall 2025 - Final Project
### Developed by Corbin Cerny

#### 1. Generate random preference profiles

Assume:
- $n$ voters
- $m ≥ 3$ alternatives
- Strict ordinal rankings

In [2]:
import itertools
import random
import numpy as np

def generate_preferences(n_voters, alternatives):
    prefs = []
    for _ in range(n_voters):
        ranking = list(alternatives)
        random.shuffle(ranking)
        prefs.append(ranking)
    return prefs

#### 2. Implement voting rules

Pairwise Majority (Condorcet)

In [3]:
def pairwise_majority(prefs, a, b):
    count = 0
    for voter in prefs:
        if voter.index(a) < voter.index(b):
            count += 1
    return count > len(prefs) / 2


Social Relation - A binary relation over alternatives that represents society’s collective ranking, derived from individual voters’ rankings.

In [4]:
def social_relation(prefs, alternatives):
    relation = {}
    for a, b in itertools.permutations(alternatives, 2):
        relation[(a, b)] = pairwise_majority(prefs, a, b)
    return relation

Borda Count for IIA

In [5]:
def borda(prefs, alternatives):
    scores = {a: 0 for a in alternatives}
    m = len(alternatives)
    for voter in prefs:
        for i, a in enumerate(voter):
            scores[a] += m - i - 1
    return sorted(scores, key=scores.get, reverse=True)


#### 3. Detect Failures

In [6]:
# Condorcet Failure Detection
def has_cycle(relation, alternatives):
    for a, b, c in itertools.permutations(alternatives, 3):
        if relation[(a,b)] and relation[(b,c)] and relation[(c,a)]:
            return True, (a,b,c)
    return False, None


In [7]:
# IIA Violation Detection
def iia_violation(prefs, base_alts, new_alt):
    ranking1 = borda(prefs, base_alts)
    ranking2 = borda(prefs, base_alts + [new_alt])
    
    def order(ranking, a, b):
        return ranking.index(a) < ranking.index(b)
    
    for a, b in itertools.combinations(base_alts, 2):
        if order(ranking1, a, b) != order(ranking2, a, b):
            return True, (a, b)
    return False, None


In [8]:
# Dictatorship Detection
def is_dictator(prefs, social_ranking):
    for i, voter in enumerate(prefs):
        if voter == social_ranking:
            return True, i
    return False, None


#### 4. Monte Carlo Experiment

In [9]:
def run_experiment(trials=1000, n_voters=7, alternatives=['A','B','C']):
    cycles = 0
    iia_failures = 0
    
    for _ in range(trials):
        prefs = generate_preferences(n_voters, alternatives)
        relation = social_relation(prefs, alternatives)
        
        has_cyc, _ = has_cycle(relation, alternatives)
        if has_cyc:
            cycles += 1
        
        iia, _ = iia_violation(prefs, ['A','B'], 'C')
        if iia:
            iia_failures += 1
    
    return {
        "cycle_rate": cycles / trials,
        "iia_failure_rate": iia_failures / trials
    }


#### 5. Visualize Results

In [ ]:
import matplotlib.pyplot as plt

def plot_condorcet_cycle(cycle):
    labels = list(cycle)
    angles = np.linspace(0, 2*np.pi, len(labels), endpoint=False)
    pos = {labels[i]: (np.cos(angles[i]), np.sin(angles[i])) for i in range(len(labels))}

    fig, ax = plt.subplots()
    for a, b in zip(labels, labels[1:] + labels[:1]):
        ax.annotate(
            "",
            xy=pos[b],
            xytext=pos[a],
            arrowprops=dict(arrowstyle="->", linewidth=2)
        )

    for node, (x, y) in pos.items():
        ax.text(x, y, node, ha='center', va='center', fontsize=12,
                bbox=dict(boxstyle="circle", fill=False))

    ax.set_aspect("equal")
    ax.axis("off")
    plt.title("Condorcet Cycle (Majority Preferences)")
    plt.show()


In [13]:
def plot_iia_violation(scores_before, scores_after):
    labels = list(scores_before.keys())
    x = np.arange(len(labels))
    width = 0.35

    fig, ax = plt.subplots()
    ax.bar(x - width/2, scores_before.values(), width, label="Before adding C")
    ax.bar(x + width/2, scores_after.values(), width, label="After adding C")

    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel("Borda Score")
    ax.set_title("IIA Violation: Social Ranking Changes")
    ax.legend()

    plt.show()


In [ ]:
def plot_cycle_rate(voter_counts, cycle_rates):
    fig, ax = plt.subplots()
    ax.plot(voter_counts, cycle_rates, marker='o')

    ax.set_xlabel("Number of Voters")
    ax.set_ylabel("Probability of Condorcet Cycle")
    ax.set_title("Condorcet Cycles Become Likely with More Voters")

    plt.show()

In [15]:
def plot_axiom_heatmap():
    rules = ["Dictatorship", "Plurality", "Borda", "Condorcet"]
    axioms = ["Pareto", "Anonymity", "Neutrality", "IIA", "Ordinality", "Decisiveness"]

    # 1 = satisfies, 0 = violates
    data = np.array([
        [1,0,0,1,1,1],  # Dictatorship
        [1,1,1,0,1,1],  # Plurality
        [1,1,1,0,1,1],  # Borda
        [1,1,1,1,1,0]   # Condorcet
    ])

    fig, ax = plt.subplots()
    im = ax.imshow(data)

    ax.set_xticks(np.arange(len(axioms)))
    ax.set_yticks(np.arange(len(rules)))
    ax.set_xticklabels(axioms)
    ax.set_yticklabels(rules)

    for i in range(len(rules)):
        for j in range(len(axioms)):
            ax.text(j, i, "✓" if data[i,j] else "✗",
                    ha="center", va="center")

    ax.set_title("Empirical Axiom Satisfaction Across Voting Rules")
    plt.show()


In [16]:
def plot_dictator_frequency(freqs):
    fig, ax = plt.subplots()
    ax.bar(range(len(freqs)), freqs)

    ax.set_xlabel("Voter Index")
    ax.set_ylabel("Frequency as Dictator")
    ax.set_title("Emergence of Dictatorship Under Arrow Constraints")

    plt.show()